In [3]:
from my_models.my_baseline_models import *
import pytorch_lightning as pl

/local/s3167445/msc_venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from losses import *

class SegmentationTask(pl.LightningModule):
    def __init__(self, model, use_focal=False, alpha=1, gamma=2, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        if use_focal:
            self.loss_fn = FocalLoss(alpha=alpha, gamma=gamma)
        else:
            self.loss_fn = nn.CrossEntropyLoss()

    def training_step(self, batch, batch_idx):
        x, y = batch
        if y.ndim == 4:
            y = torch.argmax(y, dim=1)

        logits = self.model(x)
        loss = self.loss_fn(logits, y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


In [ ]:
from data_loader import SegmentationDataModule


# --- Execution ---
resnet = ResNet18UNet(in_channels=7, num_classes=4)
task = SegmentationTask(resnet, use_focal=True)

# Setup DataModule
dm = SegmentationDataModule(root_dir='/local/s3167445/data', batch_size=16, num_workers=8)
dm.setup(stage='test')  # load test dataset

# Train on your RTX 3090s
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=[0], # Use GPU 0 as identified in your nvidia-smi
    max_epochs=30,
    precision=32#"16-mixed"

)

trainer.fit(task, datamodule=dm)
torch.save(resnet.state_dict(), "my_models/not_pretrained_resnet.pt")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/local/s3167445/msc_venv/lib64/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/local/s3167445/msc_venv/lib64/python3.9/site-packages/pytorch_lightning/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. 

Epoch 14:  84%|████████▍ | 287/341 [00:23<00:04, 12.43it/s, v_num=3, train_loss=0.0655]

In [ ]:
from torch.utils.data import DataLoader
from evaluation_functions import compute_miou
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet.to(device)  
resnet.eval()    
temp_ds = dm.test_dataset
temp_loader = DataLoader(temp_ds, batch_size=2, num_workers=4)

current_miou = compute_miou(resnet, temp_loader,num_classes=4,device=device)
print(current_miou)


0.87088543176651


In [ ]:
from my_models.my_baseline_models import *
from data_loader import SegmentationDataModule

# ---- Train EfficientNet-B0 UNet ----
efficientnet = EfficientNetB0UNet(in_channels=7, num_classes=4)
task = SegmentationTask(efficientnet, use_focal=True)

# Setup DataModule
dm = SegmentationDataModule(
    root_dir='/local/s3167445/data',
    batch_size=16,
    num_workers=8
)
dm.setup(stage='fit')

trainer = pl.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=30,
    precision=32#"16-mixed"
)

trainer.fit(task, datamodule=dm)

torch.save(efficientnet.state_dict(), "my_models/efficientnetb0_30epochs.pt")


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/local/s3167445/msc_venv/lib64/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning

Epoch 29: 100%|██████████| 341/341 [00:24<00:00, 14.13it/s, v_num=1, train_loss=0.0447] 

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 341/341 [00:24<00:00, 13.94it/s, v_num=1, train_loss=0.0447]


In [5]:


# ---- Train MobileOne-S0 UNet ----
mobilenet = MobileOneS0UNet(in_channels=7, num_classes=4)
task = SegmentationTask(mobilenet, use_focal=True)

# Setup DataModule
dm = SegmentationDataModule(
    root_dir='/local/s3167445/data',
    batch_size=16,
    num_workers=8
)
dm.setup(stage='fit')

trainer = pl.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=30,
    precision=32#"16-mixed"
)

trainer.fit(task, datamodule=dm)

torch.save(mobilenet.state_dict(), "my_models/mobileone_s0_20epochs.pt")


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name    | Type            | Params | Mode  | FLOPs
------------------------------------------------------------
0 | model   | MobileOneS0UNet | 6.9 M  | train | 0    
1 | loss_fn | FocalLoss       | 0      | train | 0    
------------------------------------------------------------
6.9 M     Trainable params
0         Non-trainable params
6.9 M     Total params
27.653    Total estimated model params size (MB)
1330      Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 29: 100%|██████████| 341/341 [00:44<00:00,  7.60it/s, v_num=2, train_loss=0.0213] 

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 341/341 [00:45<00:00,  7.51it/s, v_num=2, train_loss=0.0213]
